In [1]:
from __future__ import print_function, division
import os
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights

import albumentations as A
from albumentations.pytorch import ToTensorV2
from skimage import io

warnings.filterwarnings("ignore")

INPUT_ROOT = "/kaggle/input/cassava-leaf-disease-classification"
SEARCH_ROOTS = [INPUT_ROOT, "/kaggle/working"]

use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")
torch.backends.cudnn.benchmark = True

model = efficientnet_b4(weights=EfficientNet_B4_Weights.IMAGENET1K_V1)
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, 5)
model = model.to(device)

checkpoint_path = None
most_recent_mtime = -1
preferred_path = None

valid_exts = (".pt", ".pth", ".ckpt")
for root_dir in SEARCH_ROOTS:
    for root, _, files in os.walk(root_dir):
        for f in files:
            if f.lower().endswith(valid_exts):
                full_path = os.path.join(root, f)
                mtime = os.path.getmtime(full_path)

                if ("best" in f.lower()) or ("final" in f.lower()):
                    if preferred_path is None or mtime > most_recent_mtime:
                        preferred_path = full_path
                        most_recent_mtime = mtime
                else:
                    if (preferred_path is None) and (mtime > most_recent_mtime):
                        most_recent_mtime = mtime
                        checkpoint_path = full_path

if preferred_path is not None:
    checkpoint_path = preferred_path

checkpoint_loaded = False
if checkpoint_path and os.path.isfile(checkpoint_path):
    try:
        state = torch.load(checkpoint_path, map_location=device)
        if isinstance(state, dict):
            for key in ["model", "state_dict", "model_state_dict", "weights"]:
                if key in state:
                    state = state[key]
                    break
            if isinstance(state, dict):
                cleaned_state = {}
                for k, v in state.items():
                    new_key = k
                    for prefix in ("module.", "model.", "net."):
                        if new_key.startswith(prefix):
                            new_key = new_key[len(prefix) :]
                    cleaned_state[new_key] = v
                state = cleaned_state
        model.load_state_dict(state, strict=False)
        checkpoint_loaded = True
        print(f"Loaded checkpoint from {checkpoint_path}")
    except Exception as e:
        print(f"Failed to load checkpoint ({e}); using ImageNet weights.")
else:
    print("Checkpoint not found; using ImageNet pretrained weights.")

model.eval()

train_csv_path = os.path.join(INPUT_ROOT, "train.csv")
train_df = pd.read_csv(train_csv_path)
most_common_label = int(train_df["label"].mode()[0])
print(f"Most common training label: {most_common_label}")




Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth
100%|██████████| 74.5M/74.5M [00:00<00:00, 116MB/s]


Checkpoint not found; using ImageNet pretrained weights.
Most common training label: 3


In [2]:
class TestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = [
            f
            for f in sorted(os.listdir(root_dir))
            if os.path.isfile(os.path.join(root_dir, f))
            and f.lower().endswith((".png", ".jpg", ".jpeg"))
        ]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()
        img_name = self.images[idx]
        img_path = os.path.join(self.root_dir, img_name)
        image = io.imread(img_path)

        if image.ndim == 2:
            image = np.stack([image] * 3, axis=-1)
        elif image.shape[2] == 4:
            image = image[:, :, :3]

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]
        return img_name, image




In [3]:
transform = A.Compose(
    [
        A.Resize(380, 380),
        A.Normalize(
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
            max_pixel_value=255.0,
            p=1.0,
        ),
        ToTensorV2(),
    ]
)

test_dir = os.path.join(INPUT_ROOT, "test_images")
if not os.path.isdir(test_dir):
    raise FileNotFoundError(f"Test images directory not found: {test_dir}")

test_dataset = TestDataset(root_dir=test_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0)

names = []
predicted = []

with torch.no_grad():
    for batch_names, batch_imgs in test_loader:
        batch_imgs = batch_imgs.to(device)

        logits_orig = model(batch_imgs)

        imgs_hflip = torch.flip(batch_imgs, dims=[3])
        logits_hflip = model(imgs_hflip)

        imgs_vflip = torch.flip(batch_imgs, dims=[2])
        logits_vflip = model(imgs_vflip)

        logits_avg = (logits_orig + logits_hflip + logits_vflip) / 3.0

        # Compute confidence and apply low‑confidence fallback
        probs = torch.nn.functional.softmax(logits_avg, dim=1)
        confidences, preds = torch.max(probs, dim=1)

        preds = preds.cpu().numpy()
        confidences = confidences.cpu().numpy()
        # Replace predictions below confidence threshold with most common label
        confidence_threshold = 0.5
        for i, conf in enumerate(confidences):
            if conf < confidence_threshold:
                preds[i] = most_common_label

        names.extend(batch_names)
        predicted.extend(preds.tolist())

assert len(names) == len(predicted), "Mismatch between image IDs and predictions"



In [4]:
submission = pd.DataFrame({"image_id": names, "label": predicted})
submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")

Submission saved to /kaggle/working/submission.csv
